In [52]:
import pandas as pd                     # 데이터프레임 처리
import numpy as np                      # 수치 연산

from sklearn.model_selection import train_test_split          # train/test 분할
from sklearn.compose import ColumnTransformer                 # 컬럼별로 다른 전처리 적용
from sklearn.preprocessing import OneHotEncoder, StandardScaler  # 범주형 인코딩 / 수치형 스케일링
from sklearn.pipeline import Pipeline                         # 전처리 + 모델을 하나로 묶음

from sklearn.linear_model import LogisticRegression           # baseline 모델 1
from sklearn.tree import DecisionTreeClassifier               # baseline 모델 2
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score,   # average_precision = PR-AUC
                             confusion_matrix, classification_report)
from sklearn.base import BaseEstimator, TransformerMixin
import time                                # 모델별 튜닝 시간 측정용
from sklearn.model_selection import GridSearchCV, StratifiedKFold   # 하이퍼파라미터 탐색 / 클래스 비율 유지 K-Fold
from sklearn.base import clone                              # 튜닝된 모델의 파라미터만 복사 (학습 결과는 제외)
from imblearn.pipeline import Pipeline as ImbPipeline       # 샘플링 단계를 넣을 수 있는 Pipeline
from imblearn.under_sampling import RandomUnderSampler      # RUS
from imblearn.over_sampling import SMOTENC                  # SMOTE-NC (수치형 + 범주형 혼합 데이터용)


# 컬럼이 많아도 '...'으로 생략하지 않고 전부 표시
pd.set_option("display.max_columns", None)
# 표가 옆으로 길어져도 줄바꿈 없이 한 줄로 표시
pd.set_option("display.width", None)

### 데이터 로드 & 타깃 변환시키기

In [40]:
RANDOM_STATE = 42   # 재현성을 위한 시드 (분할, 모델 모두 동일하게 사용)

# utf-8-sig: 파일 맨 앞의 BOM 문자를 제거
# (안 하면 첫 컬럼명이 'age'가 아니라 '\ufeffage'로 읽힘)
df = pd.read_csv("add_bank.csv", encoding="utf-8-sig")

# 타깃을 yes → 1, no → 0 으로 변환
# precision/recall 등의 지표는 기본적으로 1(양성)을 기준으로 계산되므로
# 우리가 관심 있는 '가입(yes)'을 1로 둔다
df["y"] = (df["y"] == "yes").astype(int)

# 클래스 비율 확인 → no 88.3% / yes 11.7% 의 불균형 데이터
df["y"].value_counts(normalize=True)

y
0    0.883015
1    0.116985
Name: proportion, dtype: float64

### 피처 정의

In [41]:
# 수치형 변수
# - duration 제외: 통화가 끝나야 알 수 있는 값이라 사전 예측에 쓰면 데이터 누수
num_cols = ["age", "balance", "day", "campaign", "pdays", "previous"]

# 범주형 변수 → 이후 One-Hot Encoding 적용
# - age_group, balance_group: 성능 향상을 위해 생성한 파생 변수
#   구간화하면 LR처럼 선형 관계만 잡는 모델도 '20대·60대 가입률이 높다' 같은
#   비선형 패턴을 학습할 수 있음
cat_cols = ["job", "marital", "education", "default", "housing", "loan",
            "contact", "month", "poutcome", "year",
            "age_group", "balance_group"]

X = df[num_cols + cat_cols].copy()   # copy(): 원본 df가 바뀌지 않도록 복사

# year는 2008/2009/2010 세 값뿐이고 크기 비교의 의미가 약하므로 범주형으로 처리
X["year"] = X["year"].astype(str)

y = df["y"]

### Train/Validation/Test 분할

In [42]:
# 비율: train 60% / validation 20% / test 20%
# - train     : 모델 학습
# - validation: 불균형 처리 방법 비교, 하이퍼파라미터 튜닝, 임계값 결정
# - test      : 최종 선택된 모델을 마지막에 '한 번만' 평가 (중간에 보면 안 됨)

# 1단계: 전체에서 test 20%를 먼저 떼어냄
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,                  # yes 비율 11.7% 유지
    random_state=RANDOM_STATE)

# 2단계: 남은 80%를 train과 validation으로 나눔
# 남은 80% 중 25%를 validation으로 → 전체 기준 0.8 × 0.25 = 20%
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,
    stratify=y_temp,             # 여기서도 stratify 필수 (y가 아니라 y_temp 기준)
    random_state=RANDOM_STATE)

# 크기와 yes 비율 확인 → 세 세트 모두 약 11.7%여야 정상
for name, target in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name:5s} | 행 수: {len(target):6d} | yes 비율: {target.mean():.4f}")

train | 행 수:  27126 | yes 비율: 0.1170
val   | 행 수:   9042 | yes 비율: 0.1170
test  | 행 수:   9043 | yes 비율: 0.1170


### 전처리 & 모델 정의

In [43]:
# handle_unknown="ignore": test에만 있는 범주가 나와도 에러 없이 0으로 처리
ohe = OneHotEncoder(handle_unknown="ignore")

# Logistic Regression용 전처리
# - 수치형: StandardScaler로 평균 0, 표준편차 1로 맞춤
#   (LR은 변수 스케일에 민감함. balance처럼 값이 큰 변수가 학습을 지배하는 것을 방지)
# - 범주형: One-Hot Encoding
prep_lr = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", ohe, cat_cols),
])

# Decision Tree용 전처리
# - 수치형: 원본 그대로 유지
# - 범주형: OneHotEncoder
# - 트리는 '어떤 값보다 큰가/작은가'로만 분기하므로 스케일링이 필요 없음 → passthrough
prep_dt = ColumnTransformer([
    ("num", "passthrough", num_cols),
    ("cat", ohe, cat_cols),
])


### 모델별 탐색 공간  만들기
> 형식: '모델이름': (모델 또는 파이프라인, 탐색할 파라미터 후보) <br>
>불균형 처리 전이므로 class_weight, scale_pose_weight 등은 넣지 않는다.

In [47]:
# 형식: "모델이름": (모델 또는 파이프라인, 탐색할 파라미터 후보)
# - 불균형 처리 전이므로 class_weight, scale_pos_weight 등은 넣지 않음
# - 목적은 '최고 성능'이 아니라 망가지지 않은 공정한 baseline이므로 후보는 적게 잡음
search_space = {
    "LogisticRegression": (
        Pipeline([("prep", prep_lr),
                  ("clf", LogisticRegression(max_iter=1000))]),
        # C: 규제 강도의 역수. 작을수록 규제가 강해져 계수가 작아짐 (과적합 방지)
        {"clf__C": [0.01, 0.1, 1, 10]},
    ),
    "DecisionTree": (
        Pipeline([("prep", prep_dt),
                  ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE))]),
        # max_depth: 트리의 최대 깊이 (None = 제한 없음)
        # min_samples_leaf: 리프 하나에 필요한 최소 샘플 수
        # - 이 데이터는 깊이 제한만으로는 과적합이 잘 안 잡히고
        #   min_samples_leaf가 과적합 방지에 더 크게 작용하므로 범위를 넓게 탐색
        # - 깊이 제한 없는 트리(None, 1)는 train에 과적합되어
        #   예측 확률이 0/1만 나오는 문제가 있었으므로, 이를 막는 값들을 충분히 포함
        {"clf__max_depth": [3, 5, 7, 10, 15, None],
         "clf__min_samples_leaf": [1, 20, 50, 100, 150, 200, 300]},
    ),
    "RandomForest": (
        Pipeline([("prep", prep_dt),
                  ("clf", RandomForestClassifier(n_estimators=300,      # 트리 개수 (많을수록 안정적, 느려짐)
                                                 random_state=RANDOM_STATE,
                                                 n_jobs=-1))]),         # CPU 코어 모두 사용
        {"clf__max_depth": [10, 20, None],
         "clf__min_samples_leaf": [1, 5, 20]},
    ),
    "XGBoost": (
        Pipeline([("prep", prep_dt),
                  ("clf", XGBClassifier(n_estimators=300,
                                        eval_metric="logloss",          # 학습 중 경고 메시지 방지
                                        random_state=RANDOM_STATE,
                                        n_jobs=-1))]),
        # max_depth: 부스팅은 얕은 트리 여러 개를 쌓는 방식이라 3~7 정도가 일반적
        # learning_rate: 트리 하나가 반영되는 정도. 작을수록 천천히, 안정적으로 학습
        {"clf__max_depth": [3, 5, 7],
         "clf__learning_rate": [0.05, 0.1]},
    ),
    "LightGBM": (
        Pipeline([("prep", prep_dt),
                  ("clf", LGBMClassifier(n_estimators=300,
                                         random_state=RANDOM_STATE,
                                         verbose=-1,                    # 학습 로그 출력 끄기
                                         n_jobs=-1))]),
        # num_leaves: LightGBM은 깊이 대신 리프 개수로 복잡도를 조절
        {"clf__num_leaves": [15, 31, 63],
         "clf__learning_rate": [0.05, 0.1]},
    ),
    "CatBoost": (
        # CatBoost는 범주형 변수를 직접 처리하므로 원핫인코딩 Pipeline 없이 원본 X를 그대로 넣음
        # cat_features(범주형 컬럼 지정)는 여기 생성자에 넣으면
        # GridSearchCV가 모델을 복제(clone)하는 과정에서 오류가 나므로
        # ⑨에서 search.fit(..., cat_features=cat_cols)로 학습할 때 전달함
        CatBoostClassifier(iterations=300,                              # 트리 개수
                           random_seed=RANDOM_STATE,
                           verbose=0),                                  # 학습 로그 출력 끄기
        # Pipeline이 아니므로 파라미터 이름에 'clf__'를 붙이지 않음
        {"depth": [4, 6, 8],
         "learning_rate": [0.05, 0.1]},
    ),
}

In [48]:
# 하이퍼파라미터 튜닝에 사용할 5-fold 교차검증 방식
# - StratifiedKFold: 5개 fold 모두 yes 비율 11.7%를 유지하며 나눔
#   (일반 KFold를 쓰면 어떤 fold에는 yes가 너무 적게 들어갈 수 있음)
# - shuffle=True: 원본 데이터가 연도순(2008→2010)으로 정렬되어 있으므로 섞은 뒤 나눔
# - random_state: 매번 같은 방식으로 나뉘도록 고정 → 모델 간 공정한 비교
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [49]:
best_models = {}    # 모델별 최적 파라미터로 학습된 모델 저장
tuning_log = []     # 모델별 튜닝 결과 기록

for name, (estimator, grid) in search_space.items():
    start = time.time()

    search = GridSearchCV(
        estimator,
        grid,
        scoring="average_precision",   # PR-AUC 기준으로 최적 조합 선택
        cv=cv,                         # ⑤-1에서 정의한 StratifiedKFold(5-fold)
        n_jobs=1,                      # 병렬 처리는 모델 내부(n_jobs=-1)에서 하므로 여기선 1
    )

    # fit할 때 모델에 추가로 넘겨줄 인자
    # - CatBoost만 어떤 컬럼이 범주형인지 알려줘야 함
    # - 나머지 모델은 Pipeline의 원핫인코딩이 처리하므로 추가 인자 없음
    # - GridSearchCV.fit에 넣은 인자는 매 fold의 모델 학습에 그대로 전달됨
    fit_params = {"cat_features": cat_cols} if name == "CatBoost" else {}

    search.fit(X_train, y_train, **fit_params)   # train 데이터로만 탐색

    best_models[name] = search.best_estimator_   # 최적 파라미터로 train 전체에 다시 학습된 모델
    tuning_log.append({
        "model": name,
        "best_params": search.best_params_,
        "cv_pr_auc": round(search.best_score_, 4),
        "time_sec": round(time.time() - start, 1),
    })
    print(f"{name:20s} 완료 | CV PR-AUC: {search.best_score_:.4f} | {time.time() - start:.0f}초")

LogisticRegression   완료 | CV PR-AUC: 0.4354 | 2초
DecisionTree         완료 | CV PR-AUC: 0.4305 | 31초
RandomForest         완료 | CV PR-AUC: 0.4577 | 52초
XGBoost              완료 | CV PR-AUC: 0.4614 | 7초
LightGBM             완료 | CV PR-AUC: 0.4595 | 10초


KeyboardInterrupt: 

### 평가 함수

In [53]:
def evaluate(name, model, X_eval, y_eval, show_detail=True, fit_params=None):
    # fit_params: fit할 때 추가로 넘길 인자
    # - None이면 자동 설정: CatBoost 단독 모델이면 cat_features 전달, 나머지는 없음
    # - 샘플링 Pipeline 안의 CatBoost는 'clf__cat_features'처럼 단계 이름을 붙여야 해서
    #   ⑬ build_model에서 만든 값을 직접 넘겨받음
    if fit_params is None:
        fit_params = {"cat_features": cat_cols} if isinstance(model, CatBoostClassifier) else {}

    # 학습은 항상 train 데이터로만 진행 + 학습 시간 측정
    # (샘플링 Pipeline이면 이 fit 안에서 train에만 샘플링이 적용됨)
    start = time.time()
    model.fit(X_train, y_train, **fit_params)
    fit_time = time.time() - start

    # pred : 임계값 0.5 기준의 클래스 예측 (0 또는 1)
    # proba: yes(1)일 확률 → ROC-AUC, PR-AUC 계산에 사용
    # 예측 시에는 샘플링이 적용되지 않고 원본 validation 그대로 들어감
    pred = model.predict(X_eval)
    proba = model.predict_proba(X_eval)[:, 1]

    if show_detail:
        # 행 = 실제값, 열 = 예측값 / 실제 yes·예측 no 칸이 FN(놓친 가입 고객)
        cm = pd.DataFrame(confusion_matrix(y_eval, pred),
                          index=["실제 no", "실제 yes"],
                          columns=["예측 no", "예측 yes"])
        print(f"\n[{name}] 혼동행렬")
        display(cm)

    return {
        "model": name,
        "accuracy": accuracy_score(y_eval, pred),
        "precision": precision_score(y_eval, pred),   # yes로 예측한 것 중 실제 yes 비율
        "recall": recall_score(y_eval, pred),         # 실제 yes 중 모델이 찾아낸 비율
        "f1": f1_score(y_eval, pred),
        "roc_auc": roc_auc_score(y_eval, proba),
        "pr_auc": average_precision_score(y_eval, proba),
        # train PR-AUC: 원본 train 기준 (샘플링 전 데이터로 계산)
        "train_pr_auc": average_precision_score(y_train, model.predict_proba(X_train)[:, 1]),
        "fit_time_sec": fit_time,
    }

In [51]:
# 튜닝된 6개 모델을 validation 기준으로 평가
# (evaluate 안에서 모델별 혼동행렬이 display(cm)로 표 형태로 출력됨)
results = pd.DataFrame(
    [evaluate(n, m, X_val, y_val) for n, m in best_models.items()]
).set_index("model")

# ⑨에서 기록한 튜닝 소요 시간을 모델 이름 기준으로 붙임
results["tuning_time_sec"] = pd.DataFrame(tuning_log).set_index("model")["time_sec"]

# 1) 전체 결과 표: PR-AUC 높은 순으로 정렬, 소수점 4자리
print("\n[튜닝된 Baseline 성능 비교 (Validation 기준)]")
display(results.sort_values("pr_auc", ascending=False).round(4))

# 2) 지표별 최고 모델 표
# - 성능 지표는 '가장 큰 값'이 최고
# - 시간은 '가장 작은 값'이 최고 (가장 빠른 모델)
# - train_pr_auc는 과적합 확인용이라 제외
score_cols = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]
time_cols = ["fit_time_sec", "tuning_time_sec"]

best = pd.concat([
    pd.DataFrame({
        "best_model": results[score_cols].idxmax(),   # 지표별로 값이 가장 큰 모델
        "score": results[score_cols].max(),
    }),
    pd.DataFrame({
        "best_model": results[time_cols].idxmin(),    # 시간별로 값이 가장 작은 모델
        "score": results[time_cols].min(),
    }),
]).round(4)

print("\n[지표별 최고 모델]")
display(best)


[LogisticRegression] 혼동행렬


,예측 no,예측 yes
실제 no,7872,112
실제 yes,831,227



[DecisionTree] 혼동행렬


,예측 no,예측 yes
실제 no,7829,155
실제 yes,766,292



[RandomForest] 혼동행렬


...

,예측 no,예측 yes
실제 no,7833,151
실제 yes,760,298



[튜닝된 Baseline 성능 비교 (Validation 기준)]


,accuracy,precision,recall,f1,roc_auc,pr_auc,train_pr_auc,fit_time_sec,tuning_time_sec
model,,,,,,,,,
XGBoost,0.9012,0.7017,0.2713,0.3913,0.8184,0.4960,0.5133,0.1558,7.4
LightGBM,0.8992,0.6637,0.2817,0.3955,0.8173,0.4959,0.6009,0.1756,10.0
RandomForest,0.8987,0.7088,0.2278,0.3448,0.8082,0.4885,0.7068,1.5625,52.2
DecisionTree,0.8981,0.6532,0.2760,0.3880,0.7995,0.4558,0.4593,0.2243,30.9
LogisticRegression,0.8957,0.6696,0.2146,0.3250,0.8014,0.4524,0.4404,0.0875,1.7



[지표별 최고 모델]


,best_model,score
accuracy,XGBoost,0.9012
precision,RandomForest,0.7088
recall,LightGBM,0.2817
f1,LightGBM,0.3955
roc_auc,XGBoost,0.8184
pr_auc,XGBoost,0.4960
fit_time_sec,LogisticRegression,0.0875
tuning_time_sec,LogisticRegression,1.7000


### 클래스 분포 확인하고 불균형 처리 방법 정의하기
1. 불균형 데이터 처리
    - random undersampling(RUS), SMOTE-NC(수치형-범주형 섞여있음, YES 샘플과 그 이웃 사이 보간해서 새로운 YES 샘플 만들지만 범주형 값은 보간하지 않고 이웃 중 가장 많은 값을 사용)
    - class-weight='balance'
    - 데이터는 그대로 두고, yes 틀렸을 때 손실을 더 크게 주어 모델이 yes를 더 신경쓰게 만드는 방법

In [63]:
# train의 클래스별 개수 (XGBoost의 scale_pos_weight 계산에도 사용)
n_neg = (y_train == 0).sum()    # no 개수
n_pos = (y_train == 1).sum()    # yes 개수
print(f"train | no: {n_neg} / yes: {n_pos} / 비율 = 1 : {n_neg / n_pos:.2f}")

# 비율 이름 → sampling_strategy 값
# sampling_strategy = 처리 후 (yes 개수 / no 개수)
# - 1:1 → 1.0   (yes와 no가 같은 수)
# - 1:2 → 0.5   (no가 yes의 2배)
# - 1:3 → 0.333 (no가 yes의 3배)
ratios = {"1:1": 1.0, "1:2": 1 / 2, "1:3": 1 / 3}

# 비교할 방법 목록 (결과 표의 열 순서로도 사용)
methods = (["class_weight"]
           + [f"RUS {r}" for r in ratios]
           + [f"SMOTENC {r}" for r in ratios])
print("비교할 방법:", methods)


def make_sampler(method):
    """방법 이름(예: 'RUS 1:2', 'SMOTENC 1:3')에 맞는 샘플러 생성"""
    kind, ratio = method.split()             # 'RUS 1:2' → 'RUS', '1:2'
    strategy = ratios[ratio]

    if kind == "RUS":
        # no(다수 클래스)를 무작위로 줄여 목표 비율을 맞춤
        return RandomUnderSampler(sampling_strategy=strategy,
                                  random_state=RANDOM_STATE)
    if kind == "SMOTENC":
        # yes(소수 클래스)를 이웃 샘플 사이 보간으로 새로 만들어 목표 비율을 맞춤
        # categorical_features: 범주형 컬럼은 보간하지 않고 이웃들 중 가장 많은 값을 사용
        # → 'admin 30% + technician 70%' 같은 존재할 수 없는 값이 생기지 않음
        return SMOTENC(categorical_features=cat_cols,
                       sampling_strategy=strategy,
                       random_state=RANDOM_STATE)
    raise ValueError(f"알 수 없는 방법: {method}")

train | no: 23953 / yes: 3173 / 비율 = 1 : 7.55
비교할 방법: ['class_weight', 'RUS 1:1', 'RUS 1:2', 'RUS 1:3', 'SMOTENC 1:1', 'SMOTENC 1:2', 'SMOTENC 1:3']


### 모델 + 불균형 처리 조합 생성 함수

In [64]:
def build_model(name, method):
    """튜닝된 모델(best_models)에 불균형 처리 방법을 적용한 새 모델과 fit 인자를 반환"""
    # clone: 튜닝으로 찾은 하이퍼파라미터는 그대로 복사하고, 학습된 상태는 비운 새 모델 생성
    # (원본 best_models는 건드리지 않음)
    base = clone(best_models[name])
    is_catboost = isinstance(base, CatBoostClassifier)

    # ── 1) class_weight: 데이터는 그대로, yes를 틀렸을 때 손실을 더 크게 ──
    if method == "class_weight":
        # Pipeline이면 모델 단계(clf)를 꺼내고, CatBoost는 그 자체가 모델
        clf = base if is_catboost else base.named_steps["clf"]

        # 모델마다 옵션 이름이 다름 (효과는 모두 'yes에 no 대비 약 7.5배 가중치')
        if name == "XGBoost":
            clf.set_params(scale_pos_weight=n_neg / n_pos)      # no 개수 / yes 개수
        elif name == "CatBoost":
            clf.set_params(auto_class_weights="Balanced")
        else:   # LogisticRegression, DecisionTree, RandomForest, LightGBM
            clf.set_params(class_weight="balanced")

        fit_params = {"cat_features": cat_cols} if is_catboost else {}
        return base, fit_params

    # ── 2) RUS / SMOTE-NC: Pipeline 맨 앞에 샘플링 단계 추가 ──
    # 순서: 샘플링 → 전처리(원핫 등) → 모델
    # SMOTE-NC는 범주형이 문자열 상태일 때 적용해야 하므로 반드시 전처리보다 앞에 둠
    steps = [("clf", base)] if is_catboost else list(base.steps)
    model = ImbPipeline([("sampler", make_sampler(method))] + steps)

    # Pipeline 안의 CatBoost에 인자를 넘길 때는 '단계이름__인자명' 형식
    fit_params = {"clf__cat_features": cat_cols} if is_catboost else {}
    return model, fit_params

In [65]:
def build_model(name, method):
    """튜닝된 모델(best_models)에 불균형 처리 방법을 적용한 새 모델과 fit 인자를 반환"""
    # clone: 튜닝으로 찾은 하이퍼파라미터는 그대로 복사하고, 학습된 상태는 비운 새 모델 생성
    # (원본 best_models는 건드리지 않음)
    base = clone(best_models[name])
    is_catboost = isinstance(base, CatBoostClassifier)

    # ── 1) class_weight: 데이터는 그대로, yes를 틀렸을 때 손실을 더 크게 ──
    if method == "class_weight":
        # Pipeline이면 모델 단계(clf)를 꺼내고, CatBoost는 그 자체가 모델
        clf = base if is_catboost else base.named_steps["clf"]

        # 모델마다 옵션 이름이 다름 (효과는 모두 'yes에 no 대비 약 7.5배 가중치')
        if name == "XGBoost":
            clf.set_params(scale_pos_weight=n_neg / n_pos)      # no 개수 / yes 개수
        elif name == "CatBoost":
            clf.set_params(auto_class_weights="Balanced")
        else:   # LogisticRegression, DecisionTree, RandomForest, LightGBM
            clf.set_params(class_weight="balanced")

        fit_params = {"cat_features": cat_cols} if is_catboost else {}
        return base, fit_params

    # ── 2) RUS / SMOTE-NC: Pipeline 맨 앞에 샘플링 단계 추가 ──
    # 순서: 샘플링 → 전처리(원핫 등) → 모델
    # SMOTE-NC는 범주형이 문자열 상태일 때 적용해야 하므로 반드시 전처리보다 앞에 둠
    steps = [("clf", base)] if is_catboost else list(base.steps)
    model = ImbPipeline([("sampler", make_sampler(method))] + steps)

    # Pipeline 안의 CatBoost에 인자를 넘길 때는 '단계이름__인자명' 형식
    fit_params = {"clf__cat_features": cat_cols} if is_catboost else {}
    return model, fit_params

In [ ]:
imb_results = []

for method in methods:
    for name in best_models:
        model, fit_params = build_model(name, method)

        # 42개 조합의 혼동행렬은 너무 많으므로 show_detail=False
        res = evaluate(name, model, X_val, y_val,
                       show_detail=False, fit_params=fit_params)
        res["method"] = method
        imb_results.append(res)

        print(f"{method:13s} | {name:20s} | PR-AUC: {res['pr_auc']:.4f} "
              f"| recall: {res['recall']:.4f} | {res['fit_time_sec']:.1f}초")

imb_df = pd.DataFrame(imb_results)

# ⑪의 튜닝된 baseline 결과를 'baseline' 방법으로 붙여서 함께 비교
# (tuning_time_sec은 불균형 처리 결과에는 없으므로 제외)
baseline_df = (results.drop(columns="tuning_time_sec", errors="ignore")
               .reset_index()
               .assign(method="baseline"))

all_results = pd.concat([baseline_df, imb_df], ignore_index=True)

class_weight  | LogisticRegression   | PR-AUC: 0.4443 | recall: 0.4417 | 0.1초
class_weight  | DecisionTree         | PR-AUC: 0.4358 | recall: 0.4002 | 0.2초
class_weight  | RandomForest         | PR-AUC: 0.4887 | recall: 0.5049 | 1.4초
class_weight  | XGBoost              | PR-AUC: 0.4909 | recall: 0.4846 | 0.2초
class_weight  | LightGBM             | PR-AUC: 0.4914 | recall: 0.4945 | 0.2초
RUS 1:1       | LogisticRegression   | PR-AUC: 0.4456 | recall: 0.4343 | 0.0초
RUS 1:1       | DecisionTree         | PR-AUC: 0.4107 | recall: 0.3913 | 0.0초
RUS 1:1       | RandomForest         | PR-AUC: 0.4724 | recall: 0.4520 | 0.4초
RUS 1:1       | XGBoost              | PR-AUC: 0.4799 | recall: 0.4622 | 0.1초
RUS 1:1       | LightGBM             | PR-AUC: 0.4661 | recall: 0.4602 | 0.1초
RUS 1:2       | LogisticRegression   | PR-AUC: 0.4462 | recall: 0.4865 | 0.0초
RUS 1:2       | DecisionTree         | PR-AUC: 0.4148 | recall: 0.4645 | 0.1초
RUS 1:2       | RandomForest         | PR-AUC: 0.4779 | recall: 

In [67]:
# 열 순서: baseline을 맨 앞에 두고 나머지 방법 순서대로
method_order = ["baseline"] + methods

# 1) 지표별 '모델 × 방법' 표
# 행 = 모델, 열 = 불균형 처리 방법 → 같은 모델에서 방법별 변화를 한눈에 비교
for metric in ["pr_auc", "recall", "precision", "f1", "fit_time_sec"]:
    table = (all_results
             .pivot(index="model", columns="method", values=metric)
             [method_order]                         # 열 순서 정렬
             .round(4))
    print(f"\n[{metric} | 모델 × 불균형 처리 방법 (Validation 기준)]")
    display(table)

# 2) 전체 조합 상위 10개 (PR-AUC 기준)
print("\n[전체 조합 상위 10개 (PR-AUC 기준)]")
display(all_results
        .sort_values("pr_auc", ascending=False)
        .head(10)
        .set_index(["model", "method"])
        .round(4))

# 3) 지표별 최고 조합 (모델 + 방법)
score_cols = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]
best_rows = []
for col in score_cols:
    row = all_results.loc[all_results[col].idxmax()]     # 그 지표가 가장 높은 행
    best_rows.append({"metric": col, "model": row["model"],
                      "method": row["method"], "score": row[col]})
# 학습 시간은 가장 짧은 조합
row = all_results.loc[all_results["fit_time_sec"].idxmin()]
best_rows.append({"metric": "fit_time_sec", "model": row["model"],
                  "method": row["method"], "score": row["fit_time_sec"]})

print("\n[지표별 최고 조합]")
display(pd.DataFrame(best_rows).set_index("metric").round(4))

# 4) 모델별로 PR-AUC가 가장 좋았던 방법
best_per_model = all_results.loc[all_results.groupby("model")["pr_auc"].idxmax(),
                                 ["model", "method", "pr_auc", "recall", "f1"]]
print("\n[모델별 최고 방법 (PR-AUC 기준)]")
display(best_per_model.set_index("model").round(4))


[pr_auc | 모델 × 불균형 처리 방법 (Validation 기준)]


method,baseline,class_weight,RUS 1:1,RUS 1:2,RUS 1:3,SMOTENC 1:1,SMOTENC 1:2,SMOTENC 1:3
model,,,,,,,,
DecisionTree,0.4558,0.4358,0.4107,0.4148,0.4295,0.3882,0.4197,0.4176
LightGBM,0.4959,0.4914,0.4661,0.4745,0.4853,0.4347,0.4582,0.4670
LogisticRegression,0.4524,0.4443,0.4456,0.4462,0.4509,0.3986,0.4151,0.4307
RandomForest,0.4885,0.4887,0.4724,0.4779,0.4838,0.4188,0.4457,0.4586
XGBoost,0.4960,0.4909,0.4799,0.4779,0.4831,0.4231,0.4471,0.4604



[recall | 모델 × 불균형 처리 방법 (Validation 기준)]


method,baseline,class_weight,RUS 1:1,RUS 1:2,RUS 1:3,SMOTENC 1:1,SMOTENC 1:2,SMOTENC 1:3
model,,,,,,,,
DecisionTree,0.2760,0.6664,0.6834,0.5284,0.4726,0.6002,0.5104,0.4735
LightGBM,0.2817,0.6541,0.6777,0.5888,0.5425,0.5340,0.4688,0.4471
LogisticRegression,0.2146,0.6711,0.6796,0.5189,0.4310,0.6323,0.4943,0.3998
RandomForest,0.2278,0.6389,0.6692,0.5813,0.5274,0.5671,0.4981,0.4490
XGBoost,0.2713,0.6673,0.6616,0.5870,0.5142,0.5747,0.4669,0.4178



[precision | 모델 × 불균형 처리 방법 (Validation 기준)]


...

accuracy  precision  recall      f1  roc_auc  \
model        method                                                       
XGBoost      baseline        0.9012     0.7017  0.2713  0.3913   0.8184   
LightGBM     baseline        0.8992     0.6637  0.2817  0.3955   0.8173   
             class_weight    0.8435     0.3975  0.6541  0.4945   0.8187   
XGBoost      class_weight    0.8339     0.3804  0.6673  0.4846   0.8189   
RandomForest class_weight    0.8534     0.4173  0.6389  0.5049   0.8096   
             baseline        0.8987     0.7088  0.2278  0.3448   0.8082   
LightGBM     RUS 1:3         0.8814     0.4940  0.5425  0.5171   0.8160   
RandomForest RUS 1:3         0.8804     0.4899  0.5274  0.5080   0.8098   
XGBoost      RUS 1:3         0.8839     0.5037  0.5142  0.5089   0.8161   
             RUS 1:1         0.8198     0.3551  0.6616  0.4622   0.8156   

                           pr_auc  train_pr_auc  fit_time_sec  
model        method                                            
XGBoost      baseline      0.4960        0.5133        0.1558  
LightGBM     baseline      0.4959        0.6009        0.1756  
             class_weight  0.4914        0.5661        0.1891  
XGBoost      class_weight  0.4909        0.5014        0.1511  
RandomForest class_weight  0.4887        0.7284        1.3817  
             baseline      0.4885        0.7068        1.5625  
LightGBM     RUS 1:3       0.4853        0.5584        0.1615  
RandomForest RUS 1:3       0.4838        0.6227        0.6713  
XGBoost      RUS 1:3       0.4831        0.4988        0.1172  
             RUS 1:1       0.4799        0.4824        0.0958


[지표별 최고 조합]


,model,method,score
metric,,,
accuracy,XGBoost,baseline,0.9012
precision,RandomForest,baseline,0.7088
recall,DecisionTree,RUS 1:1,0.6834
f1,XGBoost,RUS 1:2,0.5194
roc_auc,XGBoost,class_weight,0.8189
pr_auc,XGBoost,baseline,0.4960
fit_time_sec,DecisionTree,RUS 1:1,0.0311



[모델별 최고 방법 (PR-AUC 기준)]


,method,pr_auc,recall,f1
model,,,,
DecisionTree,baseline,0.4558,0.2760,0.3880
LightGBM,baseline,0.4959,0.2817,0.3955
LogisticRegression,baseline,0.4524,0.2146,0.3250
RandomForest,class_weight,0.4887,0.6389,0.5049
XGBoost,baseline,0.4960,0.2713,0.3913
